In [1]:
!pip install transformers torch pandas tqdm


In [2]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Path folder di Google Drive
folder_path = '/content/drive/MyDrive/SKRIPSI MANTAP/data sosmed/.Process/Clean data'

# Path file
file_path = f'{folder_path}/twt_clean.csv'

# Membaca CSV
df = pd.read_csv(
    file_path,
    sep=',',  # gunakan ',' jika disimpan dengan to_csv() default
    engine='python'
)

# Cek nama kolom
print(df.columns)

# Ambil kolom comment
texts = df["twt_clean"].astype(str).tolist()

print("Jumlah komentar:", len(texts))

Index(['twt_clean'], dtype='object')
Jumlah komentar: 2128


In [5]:
model_name = "indolem/indobertweet-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# Gunakan GPU jika tersedia
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()


config.json:   0%|          | 0.00/1.10k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/235k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/445M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: indolem/indobertweet-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(31923, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [6]:
def indobertweet_embedding(text_list, max_length=50):
    embeddings = []

    for text in tqdm(text_list):
        encoded = tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        encoded = {k: v.to(device) for k, v in encoded.items()}

        with torch.no_grad():
            output = model(**encoded)

        # Mean Pooling
        token_embeddings = output.last_hidden_state
        attention_mask = encoded["attention_mask"]

        mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        masked_embeddings = token_embeddings * mask

        summed = torch.sum(masked_embeddings, dim=1)
        counts = torch.clamp(mask.sum(dim=1), min=1e-9)

        mean_pooled = summed / counts
        embeddings.append(mean_pooled.squeeze().cpu().numpy())

    return embeddings


In [7]:
embeddings = indobertweet_embedding(texts)

print("Jumlah embedding:", len(embeddings))
print("Dimensi embedding:", embeddings[0].shape)


100%|██████████| 2128/2128 [09:23<00:00,  3.78it/s]

Jumlah embedding: 2128
Dimensi embedding: (768,)


In [8]:
embedding_df = pd.DataFrame(embeddings)
embedding_df.columns = [f"emb_{i}" for i in range(embedding_df.shape[1])]

# Gabungkan dengan data asli
final_df = pd.concat([df.reset_index(drop=True), embedding_df], axis=1)

# Simpan hasil embedding
final_df.to_csv("twt_embed_lightgbm.csv", index=False)

print("Embedding berhasil disimpan!")


Embedding berhasil disimpan!


In [9]:
!zip twt_embed_lightgbm.zip twt_embed_lightgbm.csv


  adding: twt_embed_lightgbm.csv (deflated 57%)


In [10]:
from google.colab import files

files.download("twt_embed_lightgbm.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
import numpy as np

np.save("twt_embed_lightgbm.npy", embeddings)
from google.colab import files

files.download("twt_embed_lightgbm.npy")



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
!zip twt_embed_lightgbm.zip twt_embed_lightgbm.npy


  adding: twt_embed_lightgbm.npy (deflated 7%)


In [13]:
import os

folder_path = '/content/drive/MyDrive/SKRIPSI MANTAP/data sosmed/Twitter'

os.makedirs(folder_path, exist_ok=True)

Simpan ke gdrive

In [ ]:
output_path = f'{folder_path}/twt_embed_lightgbm.csv'

final_df.to_csv(output_path, index=False)

print("File berhasil disimpan di:")
print(output_path)